# MotoGP Grand Prix Events Held - Data Preparation

This notebook cleans and standardizes the events held dataset based on insights from the exploration phase.

## 0. Setup and Data Loading

In [ ]:
# Import necessary libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configure plot styles
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load raw data
raw_data_path = Path("../data/raw/grand_prix_events_held.csv")
df_raw = pd.read_csv(raw_data_path)

print(f"Raw data loaded: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

## 1. Data Quality Assessment

In [ ]:
# Assess data quality
print("=== DATA QUALITY ASSESSMENT ===")
print(f"Total track records: {len(df_raw)}")
print(f"Unique tracks: {df_raw['Track'].nunique()}")
print(f"Unique countries: {df_raw['Country'].nunique()}")
print(f"Total events hosted: {df_raw['Times'].sum():,}")
print(f"Events range: {df_raw['Times'].min()} - {df_raw['Times'].max()}")

print(f"\nMissing values:")
print(df_raw.isnull().sum())

print(f"\nData types:")
print(df_raw.dtypes)

print(f"\nBasic statistics:")
print(df_raw.describe())

## 2. Column Standardization

In [ ]:
# Create working copy and standardize columns
df = df_raw.copy()

# Standardize column names
column_mapping = {
    'Times': 'events_hosted',
    'Track': 'track',
    'Country': 'country'
}

df = df.rename(columns=column_mapping)

print("Standardized columns:")
print(list(df.columns))
df.head()

## 3. Data Type Corrections

In [ ]:
# Ensure proper data types
df['events_hosted'] = df['events_hosted'].astype(int)

# Clean text fields
df['track'] = df['track'].astype(str).str.strip()
df['country'] = df['country'].astype(str).str.strip().str.upper()

print("Updated data types:")
print(df.dtypes)

print(f"\nEvents hosted range after cleaning: {df['events_hosted'].min()} - {df['events_hosted'].max()}")

## 4. Track Name Standardization

In [ ]:
# Standardize track names
print("=== TRACK NAME STANDARDIZATION ===")
print(f"Original unique tracks: {df['track'].nunique()}")

# Clean track names
df['track_clean'] = df['track'].str.strip()

# Show tracks by frequency
track_distribution = df.nlargest(20, 'events_hosted')[['track_clean', 'country', 'events_hosted']]
print(f"\nTop 20 tracks by events hosted:")
print(track_distribution.to_string(index=False))

# Manual corrections for common inconsistencies if found
track_corrections = {
    # Add specific corrections if needed based on the data
    # Example: 'Circuit name variation': 'Standard Circuit Name'
}

if track_corrections:
    df['track_clean'] = df['track_clean'].replace(track_corrections)
    print(f"Applied {len(track_corrections)} track name corrections")

print(f"\nFinal unique tracks: {df['track_clean'].nunique()}")

## 5. Country Standardization and Continental Mapping

In [ ]:
# Standardize countries and add continental mapping
print("=== COUNTRY STANDARDIZATION AND CONTINENTAL MAPPING ===")

df['country_clean'] = df['country'].str.strip().str.upper()

# Show country distribution
print(f"Country distribution:")
country_summary = df.groupby('country_clean').agg({
    'events_hosted': 'sum',
    'track': 'count'
}).rename(columns={'track': 'number_of_tracks'}).sort_values('events_hosted', ascending=False)

print(country_summary.head(15))

# Comprehensive continent mapping
continent_mapping = {
    # Europe
    'NL': 'Europe', 'CZ': 'Europe', 'BE': 'Europe', 'DE': 'Europe', 'ES': 'Europe',
    'IT': 'Europe', 'FR': 'Europe', 'GB': 'Europe', 'AT': 'Europe', 'FI': 'Europe',
    'SE': 'Europe', 'CH': 'Europe', 'PT': 'Europe', 'IE': 'Europe', 'NO': 'Europe',
    'DK': 'Europe', 'HU': 'Europe', 'SI': 'Europe', 'SK': 'Europe', 'HR': 'Europe',
    'BG': 'Europe', 'RO': 'Europe', 'GR': 'Europe', 'PL': 'Europe', 'EE': 'Europe',
    'LV': 'Europe', 'LT': 'Europe', 'LU': 'Europe', 'MC': 'Europe', 'SM': 'Europe',
    'VA': 'Europe', 'LI': 'Europe', 'AD': 'Europe', 'MT': 'Europe', 'CY': 'Europe',
    
    # Asia
    'JP': 'Asia', 'MY': 'Asia', 'TH': 'Asia', 'CN': 'Asia', 'IN': 'Asia', 'ID': 'Asia',
    'KR': 'Asia', 'TW': 'Asia', 'PH': 'Asia', 'SG': 'Asia', 'VN': 'Asia', 'QA': 'Asia',
    'AE': 'Asia', 'SA': 'Asia', 'IL': 'Asia', 'TR': 'Asia', 'LB': 'Asia', 'IQ': 'Asia',
    'IR': 'Asia', 'AF': 'Asia', 'PK': 'Asia', 'BD': 'Asia', 'LK': 'Asia', 'MN': 'Asia',
    'KZ': 'Asia', 'UZ': 'Asia', 'KG': 'Asia', 'TJ': 'Asia', 'TM': 'Asia',
    
    # Oceania
    'AU': 'Oceania', 'NZ': 'Oceania', 'PG': 'Oceania', 'FJ': 'Oceania',
    'VU': 'Oceania', 'SB': 'Oceania', 'NC': 'Oceania', 'PF': 'Oceania',
    
    # North America
    'US': 'North America', 'CA': 'North America', 'MX': 'North America',
    'GT': 'North America', 'BZ': 'North America', 'CR': 'North America',
    'PA': 'North America', 'HN': 'North America', 'NI': 'North America',
    'SV': 'North America', 'JM': 'North America', 'CU': 'North America',
    'DO': 'North America', 'HT': 'North America', 'TT': 'North America',
    
    # South America
    'BR': 'South America', 'AR': 'South America', 'VE': 'South America',
    'CO': 'South America', 'PE': 'South America', 'EC': 'South America',
    'UY': 'South America', 'PY': 'South America', 'BO': 'South America',
    'CL': 'South America', 'GY': 'South America', 'SR': 'South America',
    'GF': 'South America',
    
    # Africa
    'ZA': 'Africa', 'EG': 'Africa', 'MA': 'Africa', 'TN': 'Africa',
    'DZ': 'Africa', 'LY': 'Africa', 'SD': 'Africa', 'KE': 'Africa',
    'GH': 'Africa', 'NG': 'Africa', 'ET': 'Africa', 'UG': 'Africa',
    'TZ': 'Africa', 'MZ': 'Africa', 'MG': 'Africa', 'AO': 'Africa'
}

df['continent'] = df['country_clean'].map(continent_mapping)
df['continent'] = df['continent'].fillna('Other')

print(f"\nContinent distribution:")
continent_summary = df.groupby('continent').agg({
    'events_hosted': 'sum',
    'track': 'count',
    'country_clean': 'nunique'
}).rename(columns={'track': 'number_of_tracks', 'country_clean': 'number_of_countries'})
print(continent_summary)

# Check for unmapped countries
unmapped = df[df['continent'] == 'Other']['country_clean'].unique()
if len(unmapped) > 0:
    print(f"\nUnmapped countries: {unmapped}")
else:
    print("\nAll countries successfully mapped to continents")

## 6. Hosting Frequency Categories

In [ ]:
# Create hosting frequency categories
print("=== HOSTING FREQUENCY CATEGORIES ===")

def categorize_hosting_frequency(events):
    if events >= 150:
        return 'Legendary Venue'
    elif events >= 100:
        return 'Historic Venue'
    elif events >= 50:
        return 'Regular Venue'
    elif events >= 20:
        return 'Frequent Venue'
    elif events >= 10:
        return 'Occasional Venue'
    elif events >= 5:
        return 'Rare Venue'
    else:
        return 'Very Rare Venue'

df['hosting_category'] = df['events_hosted'].apply(categorize_hosting_frequency)

# Show category distribution
category_stats = df.groupby('hosting_category').agg({
    'track': 'count',
    'events_hosted': ['sum', 'mean', 'min', 'max']
})

category_stats.columns = ['num_tracks', 'total_events', 'avg_events', 'min_events', 'max_events']
category_stats = category_stats.reindex([
    'Legendary Venue', 'Historic Venue', 'Regular Venue', 'Frequent Venue',
    'Occasional Venue', 'Rare Venue', 'Very Rare Venue'
])

print("Hosting category statistics:")
print(category_stats)

# Show examples from each category
print(f"\nExamples by category:")
for category in category_stats.index:
    if category in df['hosting_category'].values:
        examples = df[df['hosting_category'] == category].nlargest(3, 'events_hosted')
        print(f"\n{category}:")
        for _, row in examples.iterrows():
            print(f"  {row['track_clean']} ({row['country_clean']}): {row['events_hosted']} events")

## 7. Country Performance Metrics

In [ ]:
# Calculate country-level performance metrics
print("=== COUNTRY PERFORMANCE METRICS ===")

country_metrics = df.groupby('country_clean').agg({
    'events_hosted': 'sum',
    'track': 'count',
    'hosting_category': lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown'
}).rename(columns={
    'events_hosted': 'total_events',
    'track': 'number_of_tracks',
    'hosting_category': 'dominant_category'
})

# Calculate average events per track by country
country_metrics['avg_events_per_track'] = country_metrics['total_events'] / country_metrics['number_of_tracks']

# Add continent information
country_metrics['continent'] = country_metrics.index.to_series().map(continent_mapping)
country_metrics['continent'] = country_metrics['continent'].fillna('Other')

# Sort by total events
country_metrics = country_metrics.sort_values('total_events', ascending=False)

print("Top 15 countries by total events hosted:")
print(country_metrics.head(15))

print(f"\nCountries with highest average events per track:")
high_efficiency = country_metrics.nlargest(10, 'avg_events_per_track')
print(high_efficiency[['total_events', 'number_of_tracks', 'avg_events_per_track']])

## 8. Data Validation

In [ ]:
# Comprehensive data validation
print("=== DATA VALIDATION ===")

# Check for negative or zero events
invalid_events = (df['events_hosted'] <= 0).sum()
if invalid_events > 0:
    print(f"Warning: {invalid_events} tracks with invalid event counts")
else:
    print("✓ All tracks have positive event counts")

# Check for missing data in required fields
required_fields = ['track_clean', 'country_clean', 'events_hosted']
for field in required_fields:
    missing = df[field].isnull().sum()
    if missing > 0:
        print(f"Warning: {missing} missing values in {field}")
    else:
        print(f"✓ No missing values in {field}")

# Check for duplicate tracks
duplicate_tracks = df['track_clean'].duplicated().sum()
if duplicate_tracks > 0:
    print(f"Warning: {duplicate_tracks} duplicate track names")
    duplicates = df[df['track_clean'].duplicated(keep=False)].sort_values('track_clean')
    print("Duplicate tracks:")
    print(duplicates[['track_clean', 'country_clean', 'events_hosted']])
else:
    print("✓ No duplicate track names")

# Validate totals
total_events_sum = df['events_hosted'].sum()
print(f"\nTotal events across all tracks: {total_events_sum:,}")
print(f"Average events per track: {total_events_sum / len(df):.1f}")

print(f"\n=== VALIDATION SUMMARY ===")
print(f"Total tracks after preparation: {len(df)}")
print(f"Unique countries: {df['country_clean'].nunique()}")
print(f"Unique continents: {df['continent'].nunique()}")
print(f"Total events hosted: {df['events_hosted'].sum():,}")
print(f"Events range: {df['events_hosted'].min()} - {df['events_hosted'].max()}")

## 9. Final Dataset Structure

In [ ]:
# Define final column order
final_columns = [
    # Core data
    'track', 'track_clean', 'country', 'country_clean', 'continent',
    'events_hosted', 'hosting_category'
]

# Create final dataset
df_final = df[final_columns].copy()

print("=== FINAL PREPARED DATASET ===")
print(f"Shape: {df_final.shape}")
print(f"Columns: {list(df_final.columns)}")

print(f"\nSample of prepared data:")
print(df_final.head(15))

print(f"\nData types:")
print(df_final.dtypes)

print(f"\nMissing values:")
print(df_final.isnull().sum())

## 10. Save Prepared Data

In [ ]:
# Create prepared data directory if it doesn't exist
prepared_data_path = Path("../../data/interim/")
prepared_data_path.mkdir(parents=True, exist_ok=True)

# Save prepared dataset
output_file = prepared_data_path / "events_held_prepared.csv"
df_final.to_csv(output_file, index=False)

print(f"Prepared dataset saved to: {output_file}")
print(f"File size: {output_file.stat().st_size:,} bytes")

# Also save country metrics for reference
country_metrics_file = prepared_data_path / "country_hosting_metrics.csv"
country_metrics.to_csv(country_metrics_file)
print(f"Country metrics saved to: {country_metrics_file}")

# Verification
df_verification = pd.read_csv(output_file)
print(f"\nVerification - loaded shape: {df_verification.shape}")
print("✓ Files saved and verified successfully")

## 11. Preparation Summary

In [ ]:
print("=== PREPARATION SUMMARY ===")
print("\n✅ COMPLETED TASKS:")
print("1. Column name standardization")
print("2. Data type corrections and validation")
print("3. Track name cleaning and standardization")
print("4. Country standardization and continental mapping")
print("5. Hosting frequency categorization")
print("6. Country performance metrics calculation")
print("7. Data validation and consistency checks")
print("8. Prepared dataset and metrics saved")

print("\n📊 DATASET IMPROVEMENTS:")
print(f"- Standardized track and country names")
print(f"- Added comprehensive continental mapping")
print(f"- Created hosting frequency categories")
print(f"- Generated country-level performance metrics")
print(f"- Validated all data relationships")

print("\n🌍 HOSTING DISTRIBUTION:")
hosting_summary = df_final['hosting_category'].value_counts()
for category, count in hosting_summary.items():
    percentage = (count / len(df_final)) * 100
    print(f"- {category}: {count} tracks ({percentage:.1f}%)")

print("\n🏁 CONTINENTAL BREAKDOWN:")
continental_breakdown = df_final.groupby('continent').agg({
    'events_hosted': 'sum',
    'track_clean': 'count'
})
total_events = df_final['events_hosted'].sum()
for continent, row in continental_breakdown.iterrows():
    events = row['events_hosted']
    tracks = row['track_clean']
    percentage = (events / total_events) * 100
    print(f"- {continent}: {tracks} tracks hosting {events:,} events ({percentage:.1f}%)")

print("\n🔍 KEY INSIGHTS:")
top_track = df_final.nlargest(1, 'events_hosted').iloc[0]
print(f"- Most active venue: {top_track['track_clean']} ({top_track['country_clean']}) with {top_track['events_hosted']} events")
top_country = df_final.groupby('country_clean')['events_hosted'].sum().nlargest(1)
print(f"- Most active country: {top_country.index[0]} with {top_country.iloc[0]:,} total events")
legendary_venues = (df_final['hosting_category'] == 'Legendary Venue').sum()
print(f"- Legendary venues (150+ events): {legendary_venues} tracks")

print("\n➡️  READY FOR ANALYSIS PHASE")
print("The prepared events held dataset is now ready for geographical and circuit analysis.")